## Data intake & preprocessing

In [ ]:
from datasets import load_dataset

hotpot = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [ ]:
hotpot["train"][0].keys()

dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])

In [ ]:
hotpot["train"][0]['context']['title']

['Radio City (Indian radio station)',
 'History of Albanian football',
 'Echosmith',
 "Women's colleges in the Southern United States",
 'First Arthur County Courthouse and Jail',
 "Arthur's Magazine",
 '2014–15 Ukrainian Hockey Championship',
 'First for Women',
 'Freeway Complex Fire',
 'William Rast']

In [ ]:
hotpot["train"][0]['context']['sentences']

[["Radio City is India's first private FM radio station and was started on 3 July 2001.",
  ' It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003).',
  ' It plays Hindi, English and regional songs.',
  ' It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007.',
  ' Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features.',
  ' The Radio station currently plays a mix of Hindi and Regional music.',
  ' Abraham Thomas is the CEO of the company.'],
 ['Football in Albania existed before the Albanian Football Federation (FSHF) was created.',
  " This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which started in 1929 (although Albania eventually had

In [ ]:
corpus = {} # {title : text}

for example in hotpot["train"]:
    title_list = example["context"]["title"]
    sentences = example["context"]["sentences"]

    for title, article_sentences in zip(title_list, sentences):
        if title not in corpus:
            corpus[title] = " ".join(article_sentences)

In [ ]:
len(corpus)

482021

In [ ]:
lengths = [len(text) for text in corpus.values()]

print(f"Documents: {len(lengths):,}")
print(f"Average characters: {sum(lengths) / len(lengths):,.0f}")
print(f"Shortest: {min(lengths):,}")
print(f"Longest: {max(lengths):,}")

Documents: 482,021
Average characters: 482
Shortest: 24
Longest: 8,268


## Indexing

In [ ]:
%pip install -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

/Users/veronika/.pyenv/versions/3.11.1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 7143.31it/s]


In [ ]:
texts = list(corpus.values())
titles = list(corpus.keys())

In [ ]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 15064/15064 [06:46<00:00, 37.05it/s]


In [ ]:
embeddings.shape

(482021, 384)

In [ ]:
%pip install faiss-cpu

In [ ]:
import faiss

#cosine-similarity
# cos(A, B) = A * B / ||A|| * ||B||
faiss.normalize_L2(embeddings) # normalize the embeddings (dividing each by its length)
index = faiss.IndexFlatIP(384) # inner product == cosine similarity, because we already normalized the vectors
index.add(embeddings)

In [ ]:
import numpy as np
np.save("embeddings_normalized.npy", embeddings)

In [ ]:
import numpy as np
embeddings = np.load("embeddings_normalized.npy")
embeddings.shape

(482021, 384)

In [ ]:
print(index.ntotal)

482021


In [ ]:
question = hotpot["train"][0]["question"]

print(question)

Which magazine was started first Arthur's Magazine or First for Women?


In [ ]:
question_embedding = model.encode([question])
faiss.normalize_L2(question_embedding)
scores, indices = index.search(question_embedding, 5) # top-5 closest neighbours

In [ ]:
scores

array([[0.7295617, 0.6732134, 0.6707276, 0.6684101, 0.6385616]],
      dtype=float32)

In [ ]:
indices

array([[     5, 179806,      7,  23277, 429800]])

In [ ]:
print(texts[indices[0][0]])

Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.  Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others.  In May 1846 it was merged into "Godey's Lady's Book".


In [ ]:
for score, idx in zip(scores[0], indices[0]):
    print(f"Score: {score}")
    print(texts[idx])
    print()

Score: 0.7295616865158081
Arthur's Magazine (1844–1846) was an American literary periodical published in Philadelphia in the 19th century.  Edited by T.S. Arthur, it featured work by Edgar A. Poe, J.H. Ingraham, Sarah Josepha Hale, Thomas G. Spear, and others.  In May 1846 it was merged into "Godey's Lady's Book".

Score: 0.6732134222984314
Arthur magazine was a bi-monthly periodical that was founded in October 2002, by publisher Laris Kreslins and editor Jay Babcock.  It received favorable attention from other periodicals such as "L.A. Weekly", "Print", "Punk Planet" and "Rolling Stone".  "Arthur" featured photography and artwork from Spike Jonze, Art Spiegelman, Susannah Breslin, Gary Panter and Godspeed You!  Black Emperor.  Arthur's regular columnists included Byron Coley, Thurston Moore, Daniel Pinchbeck, Paul Cullum, Douglas Rushkoff, and T-Model Ford.

Score: 0.6707276105880737
First for Women is a woman's magazine published by Bauer Media Group in the USA.  The magazine was sta

### Retrieval baseline

In [ ]:
%pip install rank-bm25


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: /Users/veronika/.pyenv/versions/3.11.1/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from rank_bm25 import BM25Okapi

tokenized_corpus = [text.lower().split() for text in texts]

bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
question = hotpot["train"][100]["question"]

tokenized_question = question.lower().split()

bm25_scores = bm25.get_scores(tokenized_question)

In [ ]:
bm25_scores

array([19.06252567, 24.48145156, 20.27389873, ...,  0.        ,
        0.        ,  0.        ], shape=(482021,))

In [ ]:
# sorting the text's indices by their scores from min to max, then reversing and taking top-5 max scores
top_indices = np.argsort(bm25_scores)[::-1][:5]

In [ ]:
question

'Prince William, Duke of Gloucester and Prince George of Denmark, were eventually monarchs of which Kingdom?'

In [ ]:
for idx in top_indices:
    print(f"Score: {bm25_scores[idx]}")
    print(titles[idx])
    print(texts[idx])
    print()

Score: 72.3541942812444
Dorothy Clement
Dorothy Clement (c. 1715 – c. 1739) was the mistress of Edward Walpole and mother of his four children, including Maria Walpole, who became Duchess of Gloucester and Edinburgh upon her marriage to Prince William Henry, Duke of Gloucester and Edinburgh.  Daughter of a Darlington postmaster, she is an ancestor of Prince William, Duke of Cambridge.

Score: 64.07896926356428
Wedding of Prince William and Catherine Middleton
The wedding of Prince William, Duke of Cambridge, and Catherine Middleton took place on 29 April 2011 at Westminster Abbey in London, United Kingdom.  The groom, Prince William, Duke of Cambridge, is the eldest son of Charles, Prince of Wales and Diana, Princess of Wales, and second, after his father, in line to succeed his paternal grandmother, Queen Elizabeth II.

Score: 63.938683906870644
Prince Georg of Hanover
Prince Georg of Hanover ("Georg Paul Christian Prinz von Hannover"), Duke of Brunswick-Lüneburg (born 9 December 1949

In [ ]:
# faiss index search of the same question

question_embedding = model.encode([question])

faiss.normalize_L2(question_embedding)

scores, indices = index.search(question_embedding, 5)

In [ ]:
question

'Prince William, Duke of Gloucester and Prince George of Denmark, were eventually monarchs of which Kingdom?'

In [ ]:
for score, idx in zip(scores[0], indices[0]):
    print(f"Score: {score}")
    print(titles[idx])
    print(texts[idx])
    print()

Score: 0.7088966369628906
Prince George of Denmark
Prince George of Denmark and Norway, Duke of Cumberland (Danish: "Jørgen" ; 2 April 165328 October 1708), was the husband of Queen Anne, who reigned over Great Britain from 1702.

Score: 0.697818398475647
Prince William, Duke of Gloucester
Prince William, Duke of Gloucester (24 July 1689 – 30 July 1700 ) was the son of Princess Anne, later Queen of England, Ireland and Scotland from 1702, and her husband, Prince George, Duke of Cumberland.  He was their only child to survive infancy.  Styled Duke of Gloucester, he was viewed by contemporaries as a Protestant champion because his birth seemed to cement the Protestant succession established in the "Glorious Revolution" that had deposed his Catholic grandfather James II the previous year.

Score: 0.6693565845489502
Prince William of Gloucester
Prince William of Gloucester (William Henry Andrew Frederick; 18 December 1941 – 28 August 1972) was a grandson of King George V of the United King

In [ ]:
# looks like dense retrieval is able to capture the semantic meaning of the question, while BM25 is strongly influenced by exact word overlap

### Comparing retrieval quality

In [ ]:
validation_texts = {}

for example in hotpot["validation"]:
    for title, sentences in zip(
        example["context"]["title"],
        example["context"]["sentences"]
    ):
        validation_texts[title] = " ".join(sentences)

validation_titles = list(validation_texts.keys())
validation_texts = list(validation_texts.values())

In [ ]:
print(len(validation_titles), len(validation_texts))

66581 66581


In [ ]:
# Faiss validation index

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

validation_embeddings = model.encode(
    validation_texts,
    batch_size=16,
    show_progress_bar=True
)

In [ ]:
import numpy as np
validation_embeddings = np.load("validation_embeddings_norm.npy")
validation_embeddings.shape

(66581, 384)

In [ ]:
validation_embeddings.shape

(66581, 384)

In [ ]:
import numpy as np
np.save("validation_embeddings.npy", validation_embeddings)

In [ ]:
import numpy as np
validation_embeddings = np.load("validation_embeddings.npy")
validation_embeddings.shape

(66581, 384)

In [ ]:
import faiss

faiss.normalize_L2(validation_embeddings)

In [ ]:
import faiss
validation_index = faiss.IndexFlatIP(384)
validation_index.add(validation_embeddings)

In [ ]:
np.save("validation_embeddings_norm.npy", validation_embeddings)

In [ ]:
# BM25 validation index

In [ ]:
from rank_bm25 import BM25Okapi

tokenized_validation_texts = [text.lower().split() for text in validation_texts]

validation_bm25 = BM25Okapi(tokenized_validation_texts)

In [ ]:
# Comparing the two: can they recall the needed articles in the top 2/5/10 results

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4454.07it/s]


In [ ]:
recall_bm25 = {2: 0, 5: 0, 10: 0}
recall_dense = {2: 0, 5: 0, 10: 0}
# What percentage of questions had all required supporting articles retrieved within the top k=2/5/10 ?

In [ ]:
for example in hotpot["validation"]:

    question = example["question"]
    gold_titles = set(example["supporting_facts"]["title"])

    # BM25
    question_tokens = question.lower().split()
    bm25_scores = validation_bm25.get_scores(question_tokens)
    bm25_indices = np.argsort(bm25_scores)[::-1][:10]
    bm25_titles = [validation_titles[i] for i in bm25_indices]

    # FAISS
    question_embedding = model.encode([question]).astype("float32")
    faiss.normalize_L2(question_embedding)
    faiss_scores, faiss_indices = validation_index.search(question_embedding, 10)
    faiss_titles = [validation_titles[i] for i in faiss_indices[0]]

    # Measure recall
    for k in [2, 5, 10]:
        if gold_titles.intersection(bm25_titles[:k]):
            recall_bm25[k] += 1

        if gold_titles.intersection(faiss_titles[:k]):
            recall_dense[k] += 1

In [ ]:
n = len(hotpot["validation"])

print("BM25")
for k in [2, 5, 10]:
    print(f"Recall_{k}: {recall_bm25[k] * 100 / n:.2f}%")

print()
print("FAISS")
for k in [2, 5, 10]:
    print(f"Recall_{k}: {recall_dense[k] * 100 / n:.2f}%")

BM25
Recall_2: 70.22%
Recall_5: 81.65%
Recall_10: 87.68%

FAISS
Recall_2: 82.59%
Recall_5: 89.99%
Recall_10: 93.44%


In [ ]:
# Faiss index search substantially outperformed BM25

### Generation baseline

In [ ]:
import time
import torch
import faiss

from sentence_transformers import SentenceTransformer
from transformers import T5Tokenizer, T5ForConditionalGeneration

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")
generator = T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 131/131 [00:00<00:00, 4748.95it/s]


In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6109.04it/s]


In [ ]:
validation_texts = [
    "Scott Derrickson is an American filmmaker, screenwriter, and director.",
    "Ed Wood was an American filmmaker, actor, writer, producer, and director.",
    "Tim Burton directed the 1994 film Ed Wood.",
    "Martin Landau played Bela Lugosi in the film Ed Wood."
]

In [ ]:
validation_embeddings = embedding_model.encode(
    validation_texts,
    convert_to_numpy=True
).astype("float32")

In [ ]:
faiss.normalize_L2(validation_embeddings)

In [ ]:
dimension = validation_embeddings.shape[1]

validation_index = faiss.IndexFlatIP(dimension)

validation_index.add(validation_embeddings)

print("Documents in index:", validation_index.ntotal)

Documents in index: 4


In [ ]:
import torch


def generate_answer(
    generator,
    tokenizer,
    prompt,
    max_new_tokens=32,
):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
    )

    generator.eval()

    with torch.no_grad():
        outputs = generator.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
            decoder_start_token_id=generator.config.decoder_start_token_id,
            eos_token_id=generator.config.eos_token_id,
            pad_token_id=generator.config.pad_token_id,
            use_cache=True,
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True,
    )

In [ ]:
question = "Were Scott Derrickson and Ed Wood of the same nationality?"

question_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True,
).astype("float32")

faiss.normalize_L2(question_embedding)

scores, indices = validation_index.search(
    question_embedding,
    2,
)

retrieved_texts = [
    validation_texts[i]
    for i in indices[0]
]

context = "\n\n".join(
    f"Document {i}: {text}"
    for i, text in enumerate(retrieved_texts, start=1)
)

prompt = f"""question: {question}

context:
{context}

answer:"""

answer = generate_answer(
    generator,
    tokenizer,
    prompt,
)

print("Question:", question)
print("Retrieved documents:")

for i, text in enumerate(retrieved_texts, start=1):
    print(f"  {i}. {text}")

print("\nGenerated answer:", answer)

Question: Were Scott Derrickson and Ed Wood of the same nationality?
Retrieved documents:
  1. Ed Wood was an American filmmaker, actor, writer, producer, and director.
  2. Scott Derrickson is an American filmmaker, screenwriter, and director.

Generated answer: American


## Evaluation

In [ ]:
!nvidia-smi

Mon Aug 31 07:38:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from datasets import load_dataset

hotpot = load_dataset(
    "hotpotqa/hotpot_qa",
    "distractor"
)

print(hotpot)

README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 27.5MB            

distractor/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})


In [ ]:
validation = hotpot["validation"]

print("Validation examples:", len(validation))

Validation examples: 7405


In [ ]:
example = validation[0]

print("Keys:")
print(example.keys())

Keys:
dict_keys(['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'])


In [ ]:
print("Question:")
print(example["question"])

print("\nGold answer:")
print(example["answer"])

print("\nSupporting facts:")
print(example["supporting_facts"])

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Supporting facts:
{'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}


In [ ]:
print("\nContext type:")
print(type(example["context"]))

print("\nNumber of context entries:")
print(len(example["context"]))



Context type:
<class 'dict'>

Number of context entries:
2


In [ ]:
example["context"]

{'title': ['Ed Wood (film)',
  'Scott Derrickson',
  'Woodson, Arkansas',
  'Tyler Bates',
  'Ed Wood',
  'Deliver Us from Evil (2014 film)',
  'Adam Collis',
  'Sinister (film)',
  'Conrad Brooks',
  'Doctor Strange (2016 film)'],
 'sentences': [['Ed Wood is a 1994 American biographical period comedy-drama film directed and produced by Tim Burton, and starring Johnny Depp as cult filmmaker Ed Wood.',
   " The film concerns the period in Wood's life when he made his best-known films as well as his relationship with actor Bela Lugosi, played by Martin Landau.",
   ' Sarah Jessica Parker, Patricia Arquette, Jeffrey Jones, Lisa Marie, and Bill Murray are among the supporting cast.'],
  ['Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.',
   ' He lives in Los Angeles, California.',
   ' He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Univer

In [ ]:
print("Context type:", type(example["context"]))

if isinstance(example["context"], dict):
    print("Context keys:", example["context"].keys())
else:
    print("Context:", example["context"])

Context type: <class 'dict'>
Context keys: dict_keys(['title', 'sentences'])


In [ ]:
print("Supporting facts type:", type(example["supporting_facts"]))
print("Supporting facts:", example["supporting_facts"])

Supporting facts type: <class 'dict'>
Supporting facts: {'title': ['Scott Derrickson', 'Ed Wood'], 'sent_id': [0, 0]}


In [ ]:
print("Supporting facts:")
for fact in example["supporting_facts"]:
    print(fact)

Supporting facts:
title
sent_id


In [ ]:
validation_texts = []
validation_titles = []

for example in validation:
    titles = example["context"]["title"]
    documents = example["context"]["sentences"]

    for title, sentences in zip(titles, documents):
        text = " ".join(sentences)

        validation_titles.append(title)
        validation_texts.append(text)

print("Documents:", len(validation_texts))
print("Titles:", len(validation_titles))

Documents: 73700
Titles: 73700


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

validation_embeddings = embedding_model.encode(
    validation_texts,
    convert_to_numpy=True,
    show_progress_bar=True
).astype("float32")

faiss.normalize_L2(validation_embeddings)

validation_index = faiss.IndexFlatIP(
    validation_embeddings.shape[1]
)

validation_index.add(validation_embeddings)

print("Documents:", validation_index.ntotal)
print("Embedding dimension:", validation_embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2304 [00:00<?, ?it/s]

Documents: 73700
Embedding dimension: 384


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd


# Of the first k documents we retrieved, how many were actually relevant?
def precision_at_k(retrieved_titles, relevant_titles, k):
    retrieved = retrieved_titles[:k]
    return sum(
        title in relevant_titles
        for title in retrieved
    ) / k

# How high in the results was the first relevant document?
def reciprocal_rank(retrieved_titles, relevant_titles):
    for rank, title in enumerate(retrieved_titles, start=1):
        if title in relevant_titles:
            return 1.0 / rank

    return 0.0

In [ ]:
N = 100

retrieval_results = []

for i in range(N):
    example = validation[i]

    question = example["question"]

    # Gold documents from HotpotQA
    relevant_titles = set(
        example["supporting_facts"]["title"]
    )

    # Embedded question
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(question_embedding)

    # Retrieve top-5
    scores, indices = validation_index.search(
        question_embedding,
        5
    )

    retrieved_titles = [
        validation_titles[idx]
        for idx in indices[0]
    ]

    retrieval_results.append({
        "question": question,
        "relevant_titles": relevant_titles,
        "retrieved_titles": retrieved_titles,
        "precision@1": precision_at_k(
            retrieved_titles,
            relevant_titles,
            1
        ),
        "precision@2": precision_at_k(
            retrieved_titles,
            relevant_titles,
            2
        ),
        "precision@5": precision_at_k(
            retrieved_titles,
            relevant_titles,
            5
        ),
        "rr": reciprocal_rank(
            retrieved_titles,
            relevant_titles
        ),
    })

In [ ]:
retrieval_df = pd.DataFrame(retrieval_results)

print("Retrieval baseline")
print("------------------")
print(f"Precision@1: {retrieval_df['precision@1'].mean():.4f}")
print(f"Precision@2: {retrieval_df['precision@2'].mean():.4f}")
print(f"Precision@5: {retrieval_df['precision@5'].mean():.4f}")
print(f"MRR:         {retrieval_df['rr'].mean():.4f}")

Retrieval baseline
------------------
Precision@1: 0.7800
Precision@2: 0.5550
Precision@5: 0.2900
MRR:         0.8363


In [ ]:
# Precision@1: 0.7800 -> The top result is relevant for 78% of questions
# Precision@2: 0.5550 -> 55.5% of the top-2 retrieved documents are relevant
# Precision@5: 0.2900 -> 29% of the top-5 retrieved documents are relevant
# MRR:         0.8363 -> The first relevant document is usually very high in the ranking

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = T5ForConditionalGeneration.from_pretrained(
    model_name
)

generator.eval()

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [ ]:
import re
from collections import Counter


def normalize_answer(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    tokens = text.split()

    # Removing articles
    tokens = [
        token
        for token in tokens
        if token not in {"a", "an", "the"}
    ]

    return tokens

# Exact Match (EM): 1 → prediction matches the gold answer, 0 → it doesn't
def exact_match(prediction, gold):
    return int(
        normalize_answer(prediction)
        == normalize_answer(gold)
    )

# F1 compares the individual words/tokens in the prediction and gold answer.
# Precision — how much of the prediction was correct?
# Recall — how much of the gold answer did we recover?
# F1 = (2 * Precision * Recall) / (Precision + Recall)
def f1_score(prediction, gold):
    prediction_tokens = normalize_answer(prediction)
    gold_tokens = normalize_answer(gold)

    if not prediction_tokens or not gold_tokens:
        return float(prediction_tokens == gold_tokens)

    common = Counter(prediction_tokens) & Counter(gold_tokens)
    overlap = sum(common.values())

    if overlap == 0:
        return 0.0

    precision = overlap / len(prediction_tokens)
    recall = overlap / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
import torch

generation_results = []

N = 100

for i in range(N):
    example = validation[i]

    question = example["question"]
    gold_answer = example["answer"]

    # 1. Retrieve top-2 documents
    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(question_embedding)

    scores, indices = validation_index.search(
        question_embedding,
        2
    )

    retrieved_texts = [
        validation_texts[idx]
        for idx in indices[0]
    ]

    # 2. Build RAG prompt
    context = "\n\n".join(
        retrieved_texts
    )

    prompt = (
        f"question: {question}\n"
        f"context: {context}\n"
        f"answer:"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    # 3. Generate answer with T5
    with torch.no_grad():
        outputs = generator.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=10,
            num_beams=1,
            do_sample=False,
            decoder_start_token_id=generator.config.decoder_start_token_id,
            eos_token_id=generator.config.eos_token_id,
            pad_token_id=generator.config.pad_token_id,
            use_cache=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # 4. Calculate EM and F1
    em = exact_match(
        prediction,
        gold_answer
    )

    f1 = f1_score(
        prediction,
        gold_answer
    )

    generation_results.append({
        "question": question,
        "gold_answer": gold_answer,
        "prediction": prediction,
        "em": em,
        "f1": f1
    })

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{N}")

Processed 10/100
Processed 20/100
Processed 30/100
Processed 40/100
Processed 50/100
Processed 60/100
Processed 70/100
Processed 80/100
Processed 90/100
Processed 100/100


In [ ]:
generation_df = pd.DataFrame(generation_results)

print("Generation baseline")
print("-------------------")
print(f"EM: {generation_df['em'].mean():.4f}")
print(f"F1: {generation_df['f1'].mean():.4f}")

Generation baseline
-------------------
EM: 0.2100
F1: 0.3017


In [ ]:
pd.set_option("display.max_colwidth", 200)

generation_df[
    [
        "question",
        "gold_answer",
        "prediction",
        "em",
        "f1"
    ]
].head(20)

,question,gold_answer,prediction,em,f1
0,Were Scott Derrickson and Ed Wood of the same nationality?,yes,American,0,0.000000
1,What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?,Chief of Protocol,Shirley Temple,0,0.000000
2,"What science fantasy young adult series, told in first person, has a set of companion books narrating the stories of enslaved worlds and alien species?",Animorphs,Multiverse: Exploring Poul Anderson's,0,0.000000
3,Are the Laleli Mosque and Esma Sultan Mansion located in the same neighborhood?,no,Laleli Mosque (Turk,0,0.000000
4,"The director of the romantic comedy ""Big Stone Gap"" is based in what New York city?","Greenwich Village, New York City",Virginia,0,0.000000
5,2014 S/S is the debut album of a South Korean boy group that was formed by who?,YG Entertainment,YG Entertainment,1,1.000000
6,Who was known by his stage name Aladin and helped organizations improve their performance as a consultant?,Eenasul Fateh,Eenasul Fateh,1,1.000000
7,The arena where the Lewiston Maineiacs played their home games can seat how many people?,"3,677 seated","1,400",0,0.000000
8,"Who is older, Annie Morton or Terry Richardson?",Terry Richardson,Brandy Lynn Clark,0,0.000000
9,Are Local H and For Against both from the United States?,yes,Local H is an American rock band originally formed by,0,0.000000


In [ ]:
# Add HotpotQA query type to the generation results
generation_df["query_type"] = [
    validation[i]["type"]
    for i in range(len(generation_df))
]

# Generation performance by query type
generation_slices = (
    generation_df
    .groupby("query_type")
    .agg(
        count=("question", "size"),
        EM=("em", "mean"),
        F1=("f1", "mean")]
    )
    .reset_index()
)

generation_slices

,query_type,count,EM,F1
0,bridge,79,0.253165,0.360770
1,comparison,21,0.047619,0.079365


In [ ]:
# Add query type to retrieval results
retrieval_df["query_type"] = [
    validation[i]["type"]
    for i in range(len(retrieval_df))
]

retrieval_slices = (
    retrieval_df
    .groupby("query_type")
    .agg(
        count=("question", "size"),
        Precision_at_1=("precision@1", "mean"),
        Precision_at_2=("precision@2", "mean"),
        Precision_at_5=("precision@5", "mean"),
        MRR=("rr", "mean")
    )
    .reset_index()
)

retrieval_slices

,query_type,count,Precision_at_1,Precision_at_2,Precision_at_5,MRR
0,bridge,79,0.759494,0.518987,0.258228,0.818143
1,comparison,21,0.857143,0.690476,0.409524,0.904762


In [ ]:
# Comparison questions actually have better retrieval performance, but dramatically worse generation performance.

In [ ]:
for query_type in ["bridge", "comparison"]:
    example = next(
        ex for ex in validation
        if ex["type"] == query_type
    )

    print("=" * 80)
    print(f"QUERY TYPE: {query_type.upper()}")
    print("=" * 80)

    print("\nQuestion:")
    print(example["question"])

    print("\nGold answer:")
    print(example["answer"])

    print("\nSupporting documents:")
    for title in example["supporting_facts"]["title"]:
        print(f"- {title}")

    print()

QUERY TYPE: BRIDGE

Question:
What government position was held by the woman who portrayed Corliss Archer in the film Kiss and Tell?

Gold answer:
Chief of Protocol

Supporting documents:
- Kiss and Tell (1945 film)
- Shirley Temple
- Shirley Temple

QUERY TYPE: COMPARISON

Question:
Were Scott Derrickson and Ed Wood of the same nationality?

Gold answer:
yes

Supporting documents:
- Scott Derrickson
- Ed Wood

